In [2]:
import feedparser
from openai import OpenAI
import datetime
from dateutil import parser as dateparser
import json
import os
from bs4 import BeautifulSoup # For cleaning HTML from summaries
from dotenv import load_dotenv

# ------------- CONFIG ----------------

# It's best practice to load API keys from environment variables.
# For local testing, you can uncomment the line below and replace with your key.
  
# Load environment variables from .env file
load_dotenv()
   
# Now the rest of your script works perfectly
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
  
if not OPENAI_API_KEY:
	raise ValueError("OPENAI_API_KEY not found. Make sure it's set in your .env file.")

# Use the modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Using gpt-4o-mini: faster, more capable, and cost-effective for this task.
# It's also highly reliable for JSON-forced output.
LLM_MODEL = "gpt-4o-mini"

# RSS feeds (you can add more)
RSS_FEEDS = {
    #"NEP-ECM": "https://nep.repec.org/nep-ecm.rdf",   # Econometrics
    #"NEP-MAC": "https://nep.repec.org/nep-mac.rdf",   # Macroeconomics
    #NEP-ENE": "https://nep.repec.org/nep-ene.rdf",   # Energy Economics
    "arXiv-econEM": "http://export.arxiv.org/rss/econ.EM",
    #"arXiv-statML": "http://export.arxiv.org/rss/stat.ML"
}

SCORE_THRESHOLD = 7.0  # Keep only papers scoring above this
DAYS_BACK = 7          # Look back this many days
OUTPUT_DIR = "newsletter" # Directory to save markdown files

# ------------- HELPER FUNCTIONS ----------------

def _get_authors(entry):
    """
    Normalizes author information from a feed entry.
    Handles both 'authors' list (preferred) and 'author' string.
    """
    if hasattr(entry, 'authors') and entry.authors:
        # Typically a list of dicts with a 'name' key
        return ', '.join(author['name'] for author in entry.authors if 'name' in author)
    if hasattr(entry, 'author'):
        return entry.author
    return "Unknown"

# ------------- CORE FUNCTIONS ----------------

def fetch_recent_papers():
    """
    Fetch recent papers (last DAYS_BACK) from RSS feeds.
    Includes robust date parsing and error handling.
    """
    print(f"Fetching papers from {len(RSS_FEEDS)} sources, looking back {DAYS_BACK} days...")
    cutoff_date = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=DAYS_BACK)
    papers = []

    for source, url in RSS_FEEDS.items():
        print(f"  - Processing {source} from {url}")
        try:
            feed = feedparser.parse(url)
            if feed.bozo:
                print(f"    Warning: Malformed feed for {source}. Error: {feed.bozo_exception}")

            found_in_feed = 0
            for entry in feed.entries:
                pub_date_str = getattr(entry, "published", getattr(entry, "updated", None))
                if not pub_date_str:
                    continue

                try:
                    pub_date = dateparser.parse(pub_date_str)
                except dateparser.ParserError:
                    # print(f"    Skipping entry due to unparseable date: {entry.title}")
                    continue

                # Ensure timezone-aware comparison
                if pub_date.tzinfo is None:
                    pub_date = pub_date.replace(tzinfo=datetime.timezone.utc)

                if pub_date < cutoff_date:
                    continue

                # Clean up summary: remove HTML tags
                summary = getattr(entry, "summary", "")
                if "<" in summary and ">" in summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(separator=' ', strip=True)

                papers.append({
                    "title": entry.title,
                    "link": entry.link,
                    "summary": summary,
                    "authors": _get_authors(entry),
                    "source": source,
                    "date": pub_date.strftime("%Y-%m-%d")
                })
                found_in_feed += 1
            print(f"    Found {found_in_feed} recent papers in {source}.")

        except Exception as e:
            print(f"    Error fetching or parsing feed {source}: {e}")
            continue
    return papers

def score_and_summarize(paper):
    """
    Sends abstract to LLM for summary and scoring based on novel economic forecasting.
    Uses the modern OpenAI client and robust error handling.
    """
    prompt = f"""
You are a researcher interested in the field of econometric and machine learning forecasting methods. Also, you are an expert curator for a newsletter focused on cutting-edge and novel forecasting methods in economics.
Your task is to evaluate the provided research paper.

Here is the paper's information:

Title: {paper['title']}
Authors: {paper['authors']}
Source: {paper['source']}
Date: {paper['date']}
Abstract: {paper['summary']}

Please perform the following two steps:

1.  **Summarize the paper:** Provide a concise summary of the paper in 2–3 sentences. Focus on the main objective, methodology, and key findings relevant to economic forecasting.
2.  **Assign a relevance score:** Give a numerical score from 1 to 10 based on its relevance to *novel economic forecasting methods*.
    *   **Score 1-3 (Low Relevance):** The paper is tangentially related or uses established methods without significant innovation in forecasting.
    *   **Score 4-6 (Moderate Relevance):** The paper offers some new insights or applies methods in a new context, but the direct impact on novel economic forecasting might be limited or incremental.
    *   **Score 7-8 (High Relevance):** The paper presents a clear methodological contribution, a novel approach, or significant empirical findings directly advancing economic forecasting techniques. It offers new tools or perspectives.
    *   **Score 9-10 (Very High Relevance):** The paper introduces groundbreaking methods, significant theoretical advancements, or highly impactful empirical applications that could revolutionize economic forecasting. It's a must-read for innovation in the field.

Return your response strictly in JSON format, as shown below:
{{
  "summary": "Your 2-3 sentence summary here.",
  "score": number
}}
"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2, # Lowered slightly for more consistent scoring
            response_format={"type": "json_object"}
        )

        content = response.choices[0].message.content
        data = json.loads(content)

        # Validate the LLM's output structure
        if "summary" not in data or "score" not in data:
            raise ValueError("LLM response missing 'summary' or 'score' key.")
        if not isinstance(data["score"], (int, float)):
            raise ValueError("LLM score is not a number.")
        
        # We will keep score as float for comparison, but can format later
        data["score"] = float(data["score"])
        return data

    except Exception as e:
        print(f"Error during LLM call for paper '{paper['title']}': {e}")
        return {"summary": f"Error during processing: {e}", "score": 0.0}

def build_markdown(papers, filename):
    """
    Create markdown file with shortlisted papers.
    Ensures output directory exists and provides a clear header.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    lines = [
        f"# EconForecasting Weekly – {datetime.date.today().strftime('%Y-%m-%d')}\n",
        "A curated list of recent papers on novel economic forecasting methods.\n",
        "## Candidate Papers\n"
    ]

    if not papers:
        lines.append("No papers met the criteria this week. Check back soon!\n")
    else:
        # Sort papers by score, descending
        papers.sort(key=lambda p: p['score'], reverse=True)
        for p in papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Relevance Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}\n")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✅ Markdown file saved: {filename}")

# ------------- MAIN ----------------

if __name__ == "__main__":
    print("Starting Economic Forecasting Newsletter Pipeline...")
    
    # 1. Fetch all recent papers
    all_papers = fetch_recent_papers()
    print(f"Found {len(all_papers)} total recent papers.")

    # 2. Deduplicate papers based on their link to avoid redundant API calls
    unique_papers = []
    seen_links = set()
    for paper in all_papers:
        if paper['link'] not in seen_links:
            unique_papers.append(paper)
            seen_links.add(paper['link'])
    
    if len(all_papers) > len(unique_papers):
        print(f"Removed {len(all_papers) - len(unique_papers)} duplicates. Processing {len(unique_papers)} unique papers.")

    # 3. Score and summarize unique papers
    shortlisted = []
    for i, paper in enumerate(unique_papers, 1):
        print(f"Processing paper {i}/{len(unique_papers)}: '{paper['title'][:70]}...'")
        result = score_and_summarize(paper)

        paper.update(result) # Add summary and score to the paper dict

        if paper["score"] >= SCORE_THRESHOLD:
            shortlisted.append(paper)
            print(f"  -> Shortlisted! Score: {paper['score']:.1f}/10")
        else:
            print(f"  -> Skipped. Score: {paper['score']:.1f}/10 (Threshold: {SCORE_THRESHOLD})")

    # 4. Build the markdown report
    today_str = datetime.date.today().strftime("%Y-%m-%d")
    output_filename = os.path.join(OUTPUT_DIR, f"econ4_weekly_{today_str}.md")
    build_markdown(shortlisted, output_filename)
    
    print("\n--- Pipeline Finished ---")
    print(f"Total unique papers processed by LLM: {len(unique_papers)}")
    print(f"Shortlisted {len(shortlisted)} papers (score ≥ {SCORE_THRESHOLD}).")

Starting Economic Forecasting Newsletter Pipeline...
Fetching papers from 1 sources, looking back 7 days...
  - Processing arXiv-econEM from http://export.arxiv.org/rss/econ.EM
    Found 4 recent papers in arXiv-econEM.
Found 4 total recent papers.
Processing paper 1/4: 'Inference on Partially Identified Parameters with Separable Nuisance P...'
  -> Shortlisted! Score: 8.0/10
Processing paper 2/4: 'The purpose of an estimator is what it does: Misspecification, estiman...'
  -> Shortlisted! Score: 7.0/10
Processing paper 3/4: 'The Bayesian Context Trees State Space Model for time series modelling...'
  -> Shortlisted! Score: 8.0/10
Processing paper 4/4: 'An Empirical Risk Minimization Approach for Offline Inverse RL and Dyn...'
  -> Shortlisted! Score: 8.0/10
✅ Markdown file saved: newsletter/econ4_weekly_2025-08-28.md

--- Pipeline Finished ---
Total unique papers processed by LLM: 4
Shortlisted 4 papers (score ≥ 7.0).
